# Notebook 01 â€” Data Pipeline & TF-IDF + SVM Baseline

**Project:** Benchmarking Modern Multilingual Transformer Models for Malay-English Code-Switched Sentiment Analysis  
**Authors:** Joshua Joenathan Thomas (25141571) | Amit Kumar Gupta (25109952)  

## What this notebook does
1. Loads and inspects the MESocSentiment corpus
2. Deduplicates the provided balanced training set
3. Creates a stratified 90/10 train/dev split (seed = 42, locked)
4. Prepares the 2,000-sample manually-annotated test set as the gold benchmark
5. Trains and evaluates a TF-IDF + LinearSVC baseline â€” **Checkpoint A**

**Key decision:** The provided `Train Set.csv` was balanced by the original authors via duplicate oversampling (3,621 samples per class). We deduplicate before splitting to avoid the same tweet appearing in both train and dev.

In [ ]:
import sys
import sys
from pathlib import Path
_cwd = Path.cwd()
for _p in [_cwd / '../src', _cwd / '../3_Source', _cwd / 'src', _cwd / '3_Source']:
    if (_p / 'config.py').exists():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise ImportError('config.py not found -- run Jupyter from the project root or notebooks/ directory')

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

from config import (
    SEED, LABEL2ID, ID2LABEL, NUM_LABELS,
    TRAIN_CSV, TEST_CSV, FULL_CSV, RESULTS_DIR,
    DEV_SIZE, TEST_LABEL_COL, SVM_CONFIG
)

# Fix all random seeds
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'baseline').mkdir(parents=True, exist_ok=True)

print('Config loaded. SEED =', SEED)

In [ ]:
import sqlite3

# ── Database Setup ──────────────────────────────────────────────────────────
# All dataset partitions are persisted to SQLite BEFORE any processing.
# This satisfies the project specification requirement for database-backed
# pipelines and provides a reproducible data-provenance audit trail.

DB_PATH = RESULTS_DIR / "mesocsentiment.db"

def store_to_db(df: "pd.DataFrame", table_name: str):
    """Persist a DataFrame to the project SQLite database."""
    conn = sqlite3.connect(DB_PATH)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    print(f"  [{table_name:20s}] -> {len(df):,} rows stored in {DB_PATH.name}")

def load_from_db(table_name: str):
    """Load a named table from the project SQLite database."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(f"SELECT * FROM "{table_name}"", conn)
    conn.close()
    return df

print(f"SQLite pipeline ready  ->  {DB_PATH}")


## 1. Load and Inspect the Corpus

In [ ]:
# Full corpus â€” for reference only (not used directly in training)
df_full = pd.read_csv(FULL_CSV)
df_full.columns = ['text', 'label']
df_full['label'] = df_full['label'].str.upper().str.strip()

print('=== Full Corpus ===')
print(f'Total samples: {len(df_full):,}')
print('\nClass distribution (original corpus):')
counts = df_full['label'].value_counts()
for cls, n in counts.items():
    print(f'  {cls:10s}: {n:6,}  ({100*n/len(df_full):.2f}%)')

# Visualise corpus imbalance
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for bar, n in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{n:,}\n({100*n/len(df_full):.1f}%)', ha='center', fontsize=10)
ax.set_title('MESocSentiment â€” Full Corpus Class Distribution (19,714 tweets)', fontsize=12)
ax.set_ylabel('Count')
ax.set_xlabel('Sentiment Class')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline' / 'corpus_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: corpus_distribution.png')

## 2. Load Training Set â€” Inspect and Deduplicate

In [ ]:
df_train_raw = pd.read_csv(TRAIN_CSV)
df_train_raw.columns = ['text', 'label']
df_train_raw['label'] = df_train_raw['label'].str.upper().str.strip()
df_train_raw['text'] = df_train_raw['text'].str.strip()

print(f'Raw train set rows:    {len(df_train_raw):,}')
print(f'Raw class distribution: {dict(df_train_raw["label"].value_counts())}')

# Deduplicate â€” keep first occurrence
df_train = df_train_raw.drop_duplicates(subset='text', keep='first').reset_index(drop=True)
removed = len(df_train_raw) - len(df_train)

print(f'\nAfter deduplication:   {len(df_train):,} unique tweets ({removed} duplicates removed)')
print(f'Unique class distribution: {dict(df_train["label"].value_counts())}')
print('\nNote: Remaining imbalance after deduplication reflects the original oversampling strategy.')
print('NEUTRAL was undersampled; NEGATIVE and POSITIVE were oversampled with duplicates.')

## 3. Stratified 90/10 Train/Dev Split

In [ ]:
from sklearn.model_selection import train_test_split

df_tr, df_dev = train_test_split(
    df_train,
    test_size=DEV_SIZE,
    random_state=SEED,
    stratify=df_train['label']
)
df_tr = df_tr.reset_index(drop=True)
df_dev = df_dev.reset_index(drop=True)

print(f'Train split: {len(df_tr):,} samples')
print(f'Dev split:   {len(df_dev):,} samples')
print(f'\nTrain class dist: {dict(df_tr["label"].value_counts())}')
print(f'Dev class dist:   {dict(df_dev["label"].value_counts())}')

# Save the split for use in transformer notebook
df_tr.to_csv(RESULTS_DIR / 'train_split.csv', index=False)
df_dev.to_csv(RESULTS_DIR / 'dev_split.csv', index=False)
print('\nSplits saved to results/train_split.csv and results/dev_split.csv')
print(f'SEED = {SEED} â€” this split is fixed for all models.')

## 4. Load Gold Test Set (Manual Labels)

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
# Columns: Tweets, Sentiment (MESocSentiment), Sentiment (Manual)
df_test_raw.columns = ['text', 'label_auto', 'label_manual']
df_test_raw['text'] = df_test_raw['text'].str.strip()
df_test_raw['label_auto']   = df_test_raw['label_auto'].str.upper().str.strip()
df_test_raw['label_manual'] = df_test_raw['label_manual'].str.upper().str.strip()

df_test = df_test_raw[['text', 'label_manual']].rename(columns={'label_manual': 'label'}).copy()

print('=== Gold Test Set (manual labels) ===')
print(f'Total: {len(df_test):,} samples')
print(f'Class distribution: {dict(df_test["label"].value_counts())}')

# Quantify label noise: auto vs manual agreement
agreement = (df_test_raw['label_auto'] == df_test_raw['label_manual']).sum()
print(f'\nAuto-vs-manual label agreement: {agreement}/{len(df_test_raw)} ({100*agreement/len(df_test_raw):.1f}%)')
print('This 33.8% disagreement rate motivates use of manual labels as the benchmark.')

# Confusion matrix: auto labels vs manual labels
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
cm = confusion_matrix(df_test_raw['label_manual'], df_test_raw['label_auto'], labels=label_order)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Label Noise: Auto-annotation vs Manual Labels\n(Test Set, n=2,000)', fontsize=11)
ax.set_xlabel('Predicted (Auto-annotation)')
ax.set_ylabel('True (Manual Labels)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline' / 'label_noise_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: label_noise_confusion.png')

## 5. Preprocessing

Following the execution plan:
- Lowercase
- Remove URLs
- **Retain hashtags** (carry core sentiment in code-switched Twitter text, e.g. #KerajaanGagal)
- Tokenize for TF-IDF (whitespace-based; HuggingFace tokenizers handle transformers separately)

In [ ]:
import re

def preprocess(text: str) -> str:
    """Lowercase, remove URLs, retain hashtags and emoji."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()        # normalise whitespace
    return text

# Apply preprocessing
df_tr['text_clean']  = df_tr['text'].apply(preprocess)
df_dev['text_clean'] = df_dev['text'].apply(preprocess)
df_test['text_clean'] = df_test['text'].apply(preprocess)

# Sanity check
print('Preprocessing examples:')
for _, row in df_tr.sample(3, random_state=SEED).iterrows():
    print(f'  RAW:   {row["text"][:80]}')
    print(f'  CLEAN: {row["text_clean"][:80]}')
    print()

In [ ]:
# ── Persist raw + preprocessed partitions to SQLite ────────────────────────
# All splits stored BEFORE any ML processing.
print("Storing partitions to SQLite...")
store_to_db(df_tr[["text", "text_clean", "label"]].assign(split="train"),  "corpus_train")
store_to_db(df_dev[["text", "text_clean", "label"]].assign(split="dev"),    "corpus_dev")
store_to_db(df_test[["text", "text_clean", "label"]].assign(split="test"),  "corpus_test")

# Round-trip verification
_check = load_from_db("corpus_train")
print(f"
Round-trip check: {len(_check):,} training rows loaded from SQLite OK")
print(_check[["text_clean", "label"]].sample(2, random_state=42))


## 6. TF-IDF + LinearSVC Baseline

**Checkpoint A** â€” a defensible result exists from here regardless of transformer outcomes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# Build pipeline
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=SVM_CONFIG['tfidf_max_features'],
        ngram_range=SVM_CONFIG['tfidf_ngram_range'],
        sublinear_tf=True   # apply log(1+tf) â€” standard for imbalanced text
    )),
    ('clf', LinearSVC(
        random_state=SVM_CONFIG['random_state'],
        max_iter=2000,
        class_weight='balanced'  # handles residual class imbalance after deduplication
    ))
])

print('Fitting TF-IDF + LinearSVC on training split...')
svm_pipeline.fit(df_tr['text_clean'], df_tr['label'])
print('Done.')

In [ ]:
# Evaluate on dev set
dev_preds = svm_pipeline.predict(df_dev['text_clean'])
dev_macro_f1 = f1_score(df_dev['label'], dev_preds, average='macro')

print('=== Dev Set Performance ===')
print(f'Macro-F1: {dev_macro_f1:.4f}')
print()
print(classification_report(df_dev['label'], dev_preds,
                              target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
                              labels=['NEGATIVE', 'NEUTRAL', 'POSITIVE']))

In [ ]:
# â”€â”€ CHECKPOINT A: Evaluate on the gold test set â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test_preds = svm_pipeline.predict(df_test['text_clean'])
test_macro_f1 = f1_score(df_test['label'], test_preds, average='macro')
test_report = classification_report(
    df_test['label'], test_preds,
    target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
    labels=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
    output_dict=True
)

print('=== CHECKPOINT A: Gold Test Set Performance (TF-IDF + SVM) ===')
print(f'Macro-F1: {test_macro_f1:.4f}')
print()
print(classification_report(
    df_test['label'], test_preds,
    target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
    labels=['NEGATIVE', 'NEUTRAL', 'POSITIVE']
))

# Save results
svm_results = {
    'model': 'TF-IDF + SVM (LinearSVC)',
    'macro_f1': round(test_macro_f1, 4),
    'per_class': {
        cls: {
            'precision': round(test_report[cls]['precision'], 4),
            'recall':    round(test_report[cls]['recall'], 4),
            'f1':        round(test_report[cls]['f1-score'], 4),
        }
        for cls in ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
    }
}
with open(RESULTS_DIR / 'baseline' / 'svm_results.json', 'w') as f:
    json.dump(svm_results, f, indent=2)
print('Results saved to results/baseline/svm_results.json')

In [ ]:
# Confusion matrix â€” gold test set
label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
cm = confusion_matrix(df_test['label'], test_preds, labels=label_order)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('TF-IDF + LinearSVC â€” Gold Test Set\n'
             f'(Macro-F1 = {test_macro_f1:.4f}, n=2,000)', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline' / 'svm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: svm_confusion_matrix.png')

In [ ]:
# Decision function margin distribution â€” useful for Phase 3 ambiguous sample selection
decision_scores = svm_pipeline.decision_function(df_tr['text_clean'])
# decision_scores shape: (n_samples, n_classes)
# margin = difference between top-2 scores (small margin = ambiguous)
sorted_scores = np.sort(decision_scores, axis=1)[:, ::-1]
margins = sorted_scores[:, 0] - sorted_scores[:, 1]

df_tr_margins = df_tr.copy()
df_tr_margins['svm_margin'] = margins
df_tr_margins.to_csv(RESULTS_DIR / 'baseline' / 'train_with_margins.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(margins, bins=60, color='#3498db', edgecolor='white', linewidth=0.3)
ax.axvline(np.percentile(margins, 10), color='red', linestyle='--',
           label=f'10th percentile (margin={np.percentile(margins, 10):.2f})')
ax.set_title('LinearSVC Decision Margin Distribution (training set)\n'
             'Low-margin samples = candidate ambiguous samples for Phase 3', fontsize=11)
ax.set_xlabel('Decision margin (top-2 class score difference)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline' / 'svm_margin_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: svm_margin_distribution.png')
print(f'Train with margins saved to: results/baseline/train_with_margins.csv')

## Summary â€” Checkpoint A

| Model | Macro-F1 | NEG-F1 | NEU-F1 | POS-F1 |
|-------|----------|--------|--------|--------|
| TF-IDF + SVM | *(see above)* | - | - | - |

**This notebook is complete. The following artefacts were saved:**
- `results/train_split.csv` â€” deduplicated training split (90%)
- `results/dev_split.csv` â€” dev split (10%), same seed as transformer runs
- `results/baseline/svm_results.json` â€” numeric results
- `results/baseline/svm_confusion_matrix.png`
- `results/baseline/corpus_distribution.png`
- `results/baseline/label_noise_confusion.png`
- `results/baseline/svm_margin_distribution.png`
- `results/baseline/train_with_margins.csv` â€” used in Phase 3 for ambiguous sample selection

**Next:** `02_transformer_finetuning.ipynb`

In [ ]:
# V2 Fix: Create clean dev/test split from gold 2,000
# The original study used a noisy (auto-labelled) dev set for checkpoint
# selection, causing transformers to memorise labelling errors instead of
# learning true linguistic patterns. This cell carves a stratified 300-sample
# clean dev set and 1,700-sample clean test set from the gold-annotated test set.
# These are used exclusively by the v2 transformer experiments (Notebook 02_v2).

from sklearn.model_selection import train_test_split as _tts
from config import CLEAN_DEV_CSV, CLEAN_TEST_CSV, CLEAN_DEV_SIZE

df_clean_dev, df_clean_test = _tts(
    df_test,
    test_size=(len(df_test) - CLEAN_DEV_SIZE) / len(df_test),
    random_state=SEED,
    stratify=df_test["label"]
)
df_clean_dev  = df_clean_dev.reset_index(drop=True)
df_clean_test = df_clean_test.reset_index(drop=True)

df_clean_dev.to_csv(CLEAN_DEV_CSV,  index=False)
df_clean_test.to_csv(CLEAN_TEST_CSV, index=False)

print(f"Clean dev  saved -> {CLEAN_DEV_CSV}")
print(f"Clean test saved -> {CLEAN_TEST_CSV}")
print(f"Sizes: dev={len(df_clean_dev)}, test={len(df_clean_test)}")
print("Clean dev label dist:", df_clean_dev["label"].value_counts().to_dict())
print("Clean test label dist:", df_clean_test["label"].value_counts().to_dict())


In [ ]:
# ── Store v2 clean partitions to SQLite ────────────────────────────────────
print("Storing v2 clean partitions to SQLite...")
store_to_db(df_clean_dev.assign(split="clean_dev"),   "corpus_clean_dev")
store_to_db(df_clean_test.assign(split="clean_test"), "corpus_clean_test")
print(f"
Database tables: train ({len(df_tr):,}), dev ({len(df_dev):,}), "
      f"test ({len(df_test):,}), clean_dev ({len(df_clean_dev):,}), "
      f"clean_test ({len(df_clean_test):,})")
print(f"DB file: {DB_PATH}  ({DB_PATH.stat().st_size/1024:.1f} KB)")


In [ ]:
# V2 Fix: Re-evaluate SVM on 1,700-sample clean test set for apple-to-apple comparison
# All v2 transformer experiments use CLEAN_TEST_CSV (1,700 samples), so the SVM
# must also be evaluated on the same set â€” otherwise comparison tables mix sample sizes.
from config import CLEAN_TEST_CSV

df_clean_test = pd.read_csv(CLEAN_TEST_CSV)
df_clean_test["text_clean"] = df_clean_test["text"].apply(preprocess)

clean_test_preds = svm_pipeline.predict(df_clean_test["text_clean"])
clean_test_macro_f1 = f1_score(df_clean_test["label"], clean_test_preds, average="macro")

print("=== SVM on 1,700-sample clean test set (v2 evaluation set) ===")
print(f"Macro-F1: {clean_test_macro_f1:.4f}")
print(classification_report(df_clean_test["label"], clean_test_preds,
      labels=["NEGATIVE","NEUTRAL","POSITIVE"], digits=4))

# Save result for v2 comparison tables
from sklearn.metrics import f1_score, classification_report
per_class = {}
for lbl in ["NEGATIVE","NEUTRAL","POSITIVE"]:
    per_class[lbl] = round(f1_score(df_clean_test["label"], clean_test_preds,
                           labels=[lbl], average="macro"), 4)

svm_v2_result = {
    "key": "svm_v2",
    "model": "TF-IDF + SVM (v2 test set)",
    "macro_f1": round(clean_test_macro_f1, 4),
    "per_class_f1": per_class,
    "test_set": "clean_test_1700",
    "n_test_samples": len(df_clean_test),
}

svm_v2_path = RESULTS_DIR / "baseline" / "svm_v2_clean_test_results.json"
with open(svm_v2_path, "w") as f:
    json.dump(svm_v2_result, f, indent=2)
print(f"Saved: {svm_v2_path}")
